# Retrieval: Cross-Encoder Re-ranking & Contextual Compression

Two **post-retrieval** steps that improve the context handed to the LLM:

1. **Re-ranking** — fetch a wide candidate set cheaply, then reorder it with a more accurate (slower) cross-encoder and keep the top few.
2. **Compression** — shrink the kept chunks to the sentences that matter, cutting noise and tokens.

```
Query → retrieve top-10 → cross-encoder → top-3 → (compress) → LLM
```

Candidate generation here is plain dense search. Hybrid BM25 + dense candidates are covered in `6.7-Retrieval-Hybrid-and-Fusion.ipynb` and plug into the same re-ranker.

In [ ]:
%pip install -q langchain langchain-classic langchain-qdrant langchain-huggingface langchain-openai qdrant-client sentence-transformers rapidfuzz tqdm

## 1. Setup

In [ ]:
import random
from statistics import mean

import numpy as np
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline, EmbeddingsFilter, LLMChainExtractor,
)
from sentence_transformers import CrossEncoder

import rag_common as rc

cfg = rc.load_config()
rc.check_services(cfg)
qa_pairs, passages = rc.load_hotpotqa(cfg)
client, embeddings, doc_store = rc.connect_qdrant(cfg)
llm_name, llm = next(iter(rc.load_llms(cfg).items()))

eval_set = rc.eval_set(qa_pairs)   # the same fixed sample every RAG notebook scores on
print(f"Evaluation set: {len(eval_set)} questions | LLM: {llm_name}")


def evaluate(label, retriever):
    """Retrieve, answer and score every question in eval_set with the given retriever."""
    examples = [{**ex, "docs": retriever.invoke(ex["question"])} for ex in eval_set]
    rows = rc.evaluate_answers(label, llm, examples)
    return label, rc.summarize(rows)


def print_table(entries):
    header = f"{'Strategy':<30}{'Docs':>6}{'Chars':>8}{'Hit%':>7}{'F1%':>7}{'Time(s)':>9}"
    print(header)
    print("-" * len(header))
    for label, s in entries:
        print(f"{label:<30}{s['num_docs']:>6.1f}{s['total_chars']:>8.0f}"
              f"{s['retrieval_hit']*100:>7.1f}{s['token_f1']*100:>7.1f}{s['inference_time']:>9.2f}")

## 2. Baseline — Dense Top-3

All retrieved text goes to the LLM unchanged. Every later strategy is compared against this.

In [ ]:
dense_k3 = doc_store.as_retriever(search_kwargs={"k": 3})
dense_k10 = doc_store.as_retriever(search_kwargs={"k": 10})

summary = [evaluate("Dense top-3 (baseline)", dense_k3)]
print_table(summary)

## 3. Cross-Encoder Re-ranking

A **bi-encoder** (the embedding model) encodes query and passage separately — fast, but approximate. A **cross-encoder** reads the query and passage *together* and scores relevance directly — far more accurate, but too slow to run over the whole corpus. So we use both: bi-encoder for recall (top-10), cross-encoder for precision (keep 3).

`cross-encoder/ms-marco-MiniLM-L-6-v2` is small and trained on MS MARCO passage ranking.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


class RerankRetriever:
    """Retrieve a wide candidate set, then keep the cross-encoder's top-k."""

    def __init__(self, base, top_k=3):
        self.base, self.top_k = base, top_k

    def invoke(self, query):
        candidates = self.base.invoke(query)
        scores = cross_encoder.predict([(query, d.page_content) for d in candidates])
        ranked = sorted(zip(scores, candidates), key=lambda pair: pair[0], reverse=True)
        return [doc for _, doc in ranked[: self.top_k]]


rerank_retriever = RerankRetriever(dense_k10, top_k=3)
summary.append(evaluate("Dense top-10 → rerank top-3", rerank_retriever))
print_table(summary)

## 4. Contextual Compression

Compressors wrap a retriever and post-process its output:

| Compressor | How | Cost |
|---|---|---|
| `EmbeddingsFilter` | Drop chunks whose similarity to the query is below a threshold | Cheap — no LLM |
| `LLMChainExtractor` | LLM keeps only query-relevant sentences from each chunk | One LLM call per chunk |
| `DocumentCompressorPipeline` | Chain them: filter first, then extract from the survivors | Best cost/precision balance |

```
Query → dense (k=5) → EmbeddingsFilter → LLMChainExtractor → LLM
```

In [ ]:
THRESHOLD = 0.55   # calibrate on your data — see the diagnostic below

dense_k5 = doc_store.as_retriever(search_kwargs={"k": 5})
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=THRESHOLD)
extractor = LLMChainExtractor.from_llm(llm)

compressors = {
    f"EmbeddingsFilter (≥{THRESHOLD})": (embeddings_filter, dense_k5),
    "LLMChainExtractor": (extractor, dense_k3),
    "Pipeline (filter → extract)": (DocumentCompressorPipeline(transformers=[embeddings_filter, extractor]), dense_k5),
}

for label, (compressor, base) in compressors.items():
    retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base)
    summary.append(evaluate(label, retriever))

print_table(summary)

In [ ]:
for label, s in summary:
    rc.record_result(label, "retrieval" if "baseline" in label else "post-retrieval",
                     hit=s["retrieval_hit"], f1=s["token_f1"], n=len(eval_set))

## 5. Choosing the Threshold

`EmbeddingsFilter` is only as good as its threshold. Too high and it discards the second-hop evidence; too low and it filters nothing. Compare the question→passage similarity of **supporting** paragraphs against random **distractors**:

In [ ]:
passage_text = {p["title"]: p["text"] for p in passages}
rng = random.Random(0)

support_sims, distractor_sims = [], []
for qa in qa_pairs[:15]:
    q_vec = np.array(embeddings.embed_query(qa["question"]))
    titles = [(t, True) for t in qa["supporting_titles"] if t in passage_text]
    titles += [(t, False) for t in rng.sample(list(passage_text), 3) if t not in qa["supporting_titles"]]
    p_vecs = embeddings.embed_documents([passage_text[t] for t, _ in titles])
    for (_, is_support), p_vec in zip(titles, p_vecs):
        p_vec = np.array(p_vec)
        sim = float(q_vec @ p_vec / (np.linalg.norm(q_vec) * np.linalg.norm(p_vec)))
        (support_sims if is_support else distractor_sims).append(sim)

print(f"{'Passage type':<14}{'Min':>7}{'Mean':>7}{'Max':>7}")
for label, sims in [("supporting", support_sims), ("distractor", distractor_sims)]:
    print(f"{label:<14}{min(sims):>7.2f}{mean(sims):>7.2f}{max(sims):>7.2f}")
print(f"\nA threshold ≤ {min(support_sims):.2f} keeps every supporting paragraph (but also lets some distractors through).")

## Summary

- **Re-ranking** is usually the best first upgrade: over-fetch with the bi-encoder, let the cross-encoder pick.
- **Compression** pays off on long, noisy chunks (PDFs, web pages). On short, answer-dense passages like these it can *remove* signal — check the Hit % column.
- Order the cheap compressor before the LLM extractor to cut latency.
- Always calibrate thresholds on your own data rather than copying a default.

Next: `6.10-RAG-Prompt-Engineering-and-Orchestration.ipynb` covers the prompt side of RAG; `6.13` puts every stage together.
